# Mission 2: Market-Maker in the Arena

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/convexpi/missions/blob/main/missions/mission_02_marketmaker/notebook.ipynb)

**Learning objectives**
- Understand the economics of market-making: earning the spread vs. adverse selection
- Build a live agent that connects to the Arena via WebSocket
- Measure fill rate, inventory risk, and PnL attribution
- Iterate on quoting logic to improve performance on the live leaderboard

---

## Background

A **market-maker** continuously posts bid and ask limit orders, earning the bid-ask spread when both sides fill. The risk: an informed trader (the Arena's *informed agent*) knows where prices are going and will hit your stale quotes — this is **adverse selection**. Good market-makers:

1. Quote a spread wide enough to cover adverse selection losses
2. Manage **inventory** — avoid accumulating a large one-sided position
3. React to fills quickly — replenish cancelled/filled quotes immediately

The Arena runs a discrete-time LOB: each tick you receive the current order book state and can submit, cancel, or hold orders. Your agent has 500ms to respond.

---

## Part 0: Setup

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q convexpi>=0.1.0 websockets


In [ ]:
import asyncio
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict

plt.style.use('seaborn-v0_8-darkgrid')
print('Ready.')

In [ ]:
# Arena connection config — update these
ARENA_WS_URL = 'ws://localhost:8765'   # or your instructor's Arena URL
AGENT_ID     = 'mm_student_01'         # unique name for you on the leaderboard
INITIAL_CASH = 100_000                  # starting capital in cents

---
## Part 1: Understand the Arena Protocol

The Arena communicates over WebSocket with JSON messages. Each tick, the server sends:

```json
{
  "type": "tick",
  "tick": 42,
  "bids": [[price, qty], ...],   // top 5 bid levels, best first
  "asks": [[price, qty], ...],   // top 5 ask levels, best first
  "last_price": 10021,           // last trade price (cents)
  "your_cash": 98500,
  "your_position": -2,           // net shares long/short
  "your_pnl": -1500,
  "fills": []                    // your fills this tick
}
```

Your agent responds with a list of order actions:

```json
[
  {"action": "submit", "side": "buy",  "price": 10010, "qty": 5},
  {"action": "submit", "side": "sell", "price": 10030, "qty": 5},
  {"action": "cancel", "order_id": "abc123"}
]
```

---
## Part 2: A Minimal Market-Maker

We'll start with the simplest possible strategy: quote a fixed spread around the midpoint.

In [ ]:
class NaiveMarketMaker:
    """
    Quotes a fixed spread of `half_spread` cents around the midpoint.
    Cancels all open orders each tick before re-quoting.
    Does NOT manage inventory.
    """
    def __init__(self, half_spread: int = 5, qty: int = 10):
        self.half_spread = half_spread
        self.qty = qty
        self.open_orders: set[str] = set()

    def on_tick(self, state: dict) -> list[dict]:
        actions = []

        # Cancel all open orders
        for oid in list(self.open_orders):
            actions.append({'action': 'cancel', 'order_id': oid})
        self.open_orders.clear()

        # Compute midpoint from best bid/ask
        bids = state.get('bids', [])
        asks = state.get('asks', [])
        if not bids or not asks:
            return actions

        best_bid = bids[0][0]
        best_ask = asks[0][0]
        mid = (best_bid + best_ask) // 2

        # Quote around mid
        actions.append({'action': 'submit', 'side': 'buy',  'price': mid - self.half_spread, 'qty': self.qty})
        actions.append({'action': 'submit', 'side': 'sell', 'price': mid + self.half_spread, 'qty': self.qty})

        return actions

    def on_fill(self, fill: dict) -> None:
        pass  # not used in this version


print('NaiveMarketMaker defined.')

---
## Part 3: Connect to the Arena and Run

The cell below connects to the Arena, runs your agent for `N_TICKS` ticks, and records the full telemetry.

In [ ]:
import websockets

async def run_agent(agent, ws_url: str, agent_id: str, n_ticks: int = 200):
    """Connect agent to Arena and collect telemetry."""
    telemetry = []
    
    try:
        async with websockets.connect(ws_url, ping_interval=20) as ws:
            # Register
            await ws.send(json.dumps({'type': 'register', 'agent_id': agent_id}))
            
            tick_count = 0
            async for raw in ws:
                msg = json.loads(raw)
                if msg.get('type') != 'tick':
                    continue
                
                t0 = time.monotonic()
                actions = agent.on_tick(msg)
                latency_ms = (time.monotonic() - t0) * 1000
                
                if actions:
                    await ws.send(json.dumps(actions))
                
                # Record state
                telemetry.append({
                    'tick':      msg['tick'],
                    'pnl':       msg.get('your_pnl', 0),
                    'position':  msg.get('your_position', 0),
                    'cash':      msg.get('your_cash', 0),
                    'last_price': msg.get('last_price', 0),
                    'spread':    (msg['asks'][0][0] - msg['bids'][0][0]) if msg.get('bids') and msg.get('asks') else 0,
                    'fills':     len(msg.get('fills', [])),
                    'latency_ms': latency_ms,
                })
                
                tick_count += 1
                if tick_count % 50 == 0:
                    print(f"  tick {msg['tick']:4d}  PnL={msg.get('your_pnl',0):+7d}¢  pos={msg.get('your_position',0):+4d}")
                
                if tick_count >= n_ticks:
                    break
    except Exception as e:
        print(f"Connection error: {e}")
        print("Make sure the Arena is running at", ws_url)
    
    return pd.DataFrame(telemetry)


# Run the naive market-maker for 200 ticks
agent = NaiveMarketMaker(half_spread=5, qty=10)
print(f"Connecting to Arena at {ARENA_WS_URL} as '{AGENT_ID}'...")
df = await run_agent(agent, ARENA_WS_URL, AGENT_ID, n_ticks=200)
print(f"\nDone. Collected {len(df)} ticks.")

---
## Part 4: Analyse Telemetry

In [ ]:
if df.empty:
    print("No data — check Arena connection above.")
else:
    fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)

    axes[0].plot(df['tick'], df['pnl'] / 100, lw=1.5, color='steelblue')
    axes[0].axhline(0, color='grey', lw=0.5)
    axes[0].set_title('PnL (dollars)')

    axes[1].plot(df['tick'], df['position'], lw=1.2, color='darkorange')
    axes[1].axhline(0, color='grey', lw=0.5)
    axes[1].set_title('Inventory (shares)')

    axes[2].plot(df['tick'], df['last_price'] / 100, lw=1, color='grey', label='Last price')
    axes[2].set_title('Market price ($)')
    axes[2].set_xlabel('Tick')

    plt.tight_layout()
    plt.show()

    total_fills = df['fills'].sum()
    final_pnl   = df['pnl'].iloc[-1] / 100
    max_inv     = df['position'].abs().max()
    print(f"\nSummary")
    print(f"  Total fills  : {total_fills}")
    print(f"  Final PnL    : ${final_pnl:.2f}")
    print(f"  Max |inventory|: {max_inv} shares")
    print(f"  Avg response latency: {df['latency_ms'].mean():.2f} ms")

### Diagnosing performance

| Symptom | Likely cause | Fix |
|---|---|---|
| PnL drifts negative despite fills | Adverse selection — informed trader hits your quotes | Widen spread or skew quotes away from inventory |
| Inventory grows monotonically | No inventory management | Add position limits and quote asymmetrically |
| Very few fills | Spread too wide | Tighten spread or increase quote size |
| High latency (>100ms) | Heavy compute in `on_tick` | Move expensive logic out of the hot path |

---
## Part 5: Inventory-Aware Market-Maker

The key improvement: skew your quotes based on current inventory. If you're long, lower your ask to encourage selling. If you're short, raise your bid to encourage buying.

In [ ]:
class InventoryAwareMarketMaker:
    """
    Quotes around mid with inventory skew.
    
    skew_per_share: cents to shift mid per share of inventory.
    max_position: cancel quotes if |inventory| exceeds this.
    """
    def __init__(self, half_spread: int = 6, qty: int = 10,
                 skew_per_share: float = 0.5, max_position: int = 50):
        self.half_spread    = half_spread
        self.qty            = qty
        self.skew_per_share = skew_per_share
        self.max_position   = max_position
        self.open_orders: set[str] = set()

    def on_tick(self, state: dict) -> list[dict]:
        actions = []

        # Cancel stale orders
        for oid in list(self.open_orders):
            actions.append({'action': 'cancel', 'order_id': oid})
        self.open_orders.clear()

        bids = state.get('bids', [])
        asks = state.get('asks', [])
        if not bids or not asks:
            return actions

        position = state.get('your_position', 0)

        # Hard inventory limit — stop quoting until flat-ish
        if abs(position) >= self.max_position:
            return actions

        best_bid = bids[0][0]
        best_ask = asks[0][0]
        mid = (best_bid + best_ask) / 2

        # Skew mid by inventory: long → lower mid → easier to sell
        skewed_mid = mid - position * self.skew_per_share

        bid_price = round(skewed_mid - self.half_spread)
        ask_price = round(skewed_mid + self.half_spread)

        # Don't cross the market
        if bid_price >= best_ask or ask_price <= best_bid:
            return actions

        # Size down as inventory grows
        size_scale = max(0.2, 1 - abs(position) / self.max_position)
        qty = max(1, round(self.qty * size_scale))

        actions.append({'action': 'submit', 'side': 'buy',  'price': bid_price, 'qty': qty})
        actions.append({'action': 'submit', 'side': 'sell', 'price': ask_price, 'qty': qty})

        return actions


print('InventoryAwareMarketMaker defined.')

In [ ]:
# Run the improved agent
agent2 = InventoryAwareMarketMaker(half_spread=6, qty=10, skew_per_share=0.5, max_position=50)
print(f"Connecting as '{AGENT_ID}_v2'...")
df2 = await run_agent(agent2, ARENA_WS_URL, AGENT_ID + '_v2', n_ticks=200)

if not df2.empty:
    print(f"\nNaive PnL:    ${df['pnl'].iloc[-1]/100:.2f}")
    print(f"Improved PnL: ${df2['pnl'].iloc[-1]/100:.2f}")

---
## Part 6: PnL Attribution

Break down where your PnL comes from: spread capture vs. inventory mark-to-market.

In [ ]:
def attribute_pnl(df: pd.DataFrame) -> pd.DataFrame:
    """Decompose PnL into spread income and inventory MTM."""
    if df.empty:
        return df
    df = df.copy()
    price_change = df['last_price'].diff().fillna(0)
    df['mtm_pnl']    = df['position'].shift(1).fillna(0) * price_change / 100
    df['spread_pnl_cumulative'] = (df['pnl'] / 100) - df['mtm_pnl'].cumsum()
    return df

if not df2.empty:
    attr = attribute_pnl(df2)
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(attr['tick'], attr['pnl'] / 100, label='Total PnL', lw=2)
    ax.plot(attr['tick'], attr['spread_pnl_cumulative'], label='Spread income', lw=1.5, linestyle='--')
    ax.plot(attr['tick'], attr['mtm_pnl'].cumsum(), label='Inventory MTM', lw=1.5, linestyle=':')
    ax.axhline(0, color='grey', lw=0.5)
    ax.set_title('PnL attribution')
    ax.set_xlabel('Tick')
    ax.legend()
    plt.tight_layout()
    plt.show()

---
## Part 7: Challenges

**Challenge A (Easy):** The `half_spread` is currently fixed. Implement a **volatility-adaptive spread**: measure the standard deviation of `last_price` over the last 20 ticks and set `half_spread = k * vol` for some constant `k`. Does this improve PnL stability?

**Challenge B (Medium):** Implement **quote stuffing protection**: if you notice you're getting adversely selected more than N times in a row (fills that move against you immediately), temporarily widen your spread for 10 ticks.

**Challenge C (Medium):** Add a **mean-reversion layer**: if your inventory is large, place an aggressive limit order to flatten it (instead of just skewing the quotes). Measure whether this reduces your inventory half-life.

**Challenge D (Hard):** Estimate the **informed-trader probability** (PIN) on each tick using the fill-rate imbalance between buy fills and sell fills. Adjust your spread dynamically: quote wider when PIN is high, tighter when it's low.

In [ ]:
# Challenge A starter — volatility-adaptive spread
class VolAdaptiveMM(InventoryAwareMarketMaker):
    def __init__(self, k: float = 0.5, **kwargs):
        super().__init__(**kwargs)
        self.k = k
        self._price_history: list[float] = []

    def on_tick(self, state: dict) -> list[dict]:
        last = state.get('last_price', 0)
        self._price_history.append(last)
        if len(self._price_history) > 20:
            self._price_history.pop(0)
        if len(self._price_history) >= 5:
            vol = float(np.std(self._price_history))
            self.half_spread = max(3, round(self.k * vol))
        return super().on_tick(state)


print('VolAdaptiveMM defined. Run it the same way as the agents above.')

---
## Wrap-up

Key takeaways:

1. **Spread is not free money.** The bid-ask spread compensates for adverse selection. If informed traders hit your quotes, the spread you earn is less than the inventory losses you incur.

2. **Inventory is risk.** A market-maker who doesn't manage inventory is just a directional trader with extra steps. Every quote is an option you sell to the market — know your Greeks.

3. **Skewing quotes is the textbook fix.** It reduces adverse selection by making it expensive for informed traders to pick off a one-sided book. But it also reduces fill rate on the "wrong" side.

4. **Volatility governs the optimal spread.** In quiet markets, narrow spreads maximise flow. In volatile markets, a narrow spread is a gift to the informed.

---
*Mission 3: Alpha Discovery → finding the planted signal in the synthetic market*